# StarCraft 2 Matchmaking Prototype

The goal is to test a new matchmaking system **without rounds** and **without divisions** (divisions may still be present in the frontend to report progress, but are not used for matchmaking itself).

The matchmaking must ensure that **all** bots

- play a "fair" number of matches;
- play predominantly matches against opponents of a similar skill;
- play matches gainst varied opponents within those of similar skill;
- play matches which are somewhat evenly distributed in time.

We need to verify that these requirements are fulfilled by any proposed matchmaking system by running it on a model of the real ladder and checking

- the match frequency distributions;
- the match-up rating-difference distributions;
- the match-up opponent distributions.

In case that the proposed matchmaking is non-greedy, i.e. has a mechanism to wait for bot to become available instead of starting a game immediately, we also need to check the ladder server utilization.

## Ladder Model

We need to simulate ratings updates and match durations in our ladder model.

### Ladder Model Ratings Updates

The expected score $E_\text{A}$ of a match between bots A and B (from A's perspective) with ratings $R_\text{A}$ and $R_\text{B}$ is given as

$$
E_\text{A} = \frac{1}{1 + 10^{\frac{R_\text{A} - R_\text{B}}{400}}}
$$

and update rule for bot A is

$$
\Delta R_\text{A} = K \cdot ( S_\text{A} - E_\text{A} ),
$$

where $K$ is the adjustment per game (set to 16 on AI Arena) and $S_\text{A}$ is A's actual score of the match (1 for a win, 0.5 for a draw and 0 for a loss).

The expected score $E_\text{A}$ is only a proxy for A's win probability if there is no possibility for a draw.
In the bot games of AI Arena, we frequently encounter draws, and we need to model them.

Additionally, ratings on AI Arena are often **not transitive**, i.e. A beating B almost surely and B beating C a.s. does not imply A beating C a.s. (which the ELO system assumes). For this reason, where we have data available we should use it to inform the outcome of a simulated match.
To get a smooth transition between no-data to data match-ups, we use the ELO system estimated (augmented with a global draw rate, $d$) as the prior and perform Bayesian inference with actual data, where available.

For the pior we use the 3-dimensional Dirichlet distribution (multivariate beta distribution) which is then updated with the observed number of wins ($W$), losses ($L$) and draws ($D$), i.e.

$$
\begin{aligned}
&P_\text{A}^\text{ELO}(\text{win}) = (1 - d) \cdot E_\text{A},\\
&P_\text{A}^\text{ELO}(\text{loss}) = (1 - d) \cdot (1 - E_\text{A}),\\
&P_\text{A}^\text{ELO}(\text{draw}) = d;
\end{aligned}
$$

$$
\text{posterior} = \text{Dir}(n_0 \cdot P_\text{A}^\text{ELO}(\text{win}) + W, \quad n_0 \cdot P_\text{A}^\text{ELO}(\text{draw}) + D, \quad n_0 \cdot P_\text{A}^\text{ELO}(\text{loss}) + L),
$$

where $n_0$ determines the strength of the prior (in units of number of matches). In the following I will use $n_0 = 10$, which means that if we have real data for 10 matches, we consider the evidence from the real data as strong as the prior.

